In [1]:
#!pip install pandas
#!pip install seaborn
#!pip install openpyxl
#!pip install yfinance

In [2]:
import pandas as pd
import numpy as np
import time
import csv
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from tensorflow.keras.layers import LSTM
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.models import model_from_json
from tensorflow import data
from matplotlib import pyplot
from sklearn.metrics import mean_squared_error
from sklearn.metrics import median_absolute_error
from sklearn.preprocessing import MinMaxScaler
from tensorflow.keras.callbacks import EarlyStopping


In [3]:
path_name_results='../results/'
file_result = 'Result_LSTM_IBM_stock_prices.csv'

In [4]:
pip install pip_system_certs


[notice] A new release of pip is available: 24.3.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [5]:
# Load CSV IBM stock prices
df_raw = pd.read_csv('../datasets/IBM_stock_prices.csv')

# show columns
print("Cols availables:", df_raw.columns.tolist())

# create dataset
dataset = pd.DataFrame()
dataset['date'] = pd.to_datetime(df_raw['Date'])
dataset['num_observations'] = df_raw['Close']  


Cols availables: ['Date', 'Open', 'High', 'Low', 'Close', 'Volume']


In [6]:
dataset.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 11702 entries, 0 to 11701
Data columns (total 2 columns):
 #   Column            Non-Null Count  Dtype         
---  ------            --------------  -----         
 0   date              11702 non-null  datetime64[ns]
 1   num_observations  11702 non-null  float64       
dtypes: datetime64[ns](1), float64(1)
memory usage: 183.0 KB


In [7]:
#checks if there are null variables
dataset.isna().sum()

date                0
num_observations    0
dtype: int64

In [8]:
def salvar_resultado(nm_dataset, ds_best_param, n_time_steps, MSE, RMSE, MAE, MAPE, sMAPE, Duration):
  #Script to write training cycle results
  data = [nm_dataset, ds_best_param, n_time_steps, MSE, RMSE, MAE, MAPE, sMAPE, Duration]
  fields = ['Dataset','Best Params','n_time_steps','MSE', 'RMSE', 'MAE', 'MAPE','sMAPE','Duration']
  with open(f'{path_name_results}{file_result}', "a",newline='') as csv_file:
    writer = csv.writer(csv_file,delimiter=';')
    writer.writerow(data)  
  print(fields)
  print(data)
    
#Script to create the results file
def criar_arquivo_resultado():
  fields = ['Dataset','Best Params','n_time_steps','MSE', 'RMSE', 'MAE','MAPE','sMAPE','Duration']
  with open(f'{path_name_results}{file_result}', "w",newline='') as csv_file:
    writer = csv.writer(csv_file,delimiter=';')
    writer.writerow(fields)    

In [9]:
# convert an array of values into a dataset matrix
def create_matrix_dataset(dataset, n_time_steps=1):
    dX, dY = [], []
    for i in range(len(dataset) - n_time_steps - 1):
        a = dataset[i:(i + n_time_steps)]
        dX.append(a)
        dY.append(dataset[i + n_time_steps])
    return np.array(dX), np.array(dY)

In [10]:
  
def save_model(model,n_time_steps):
  # serialize model to JSON
  model_json = model.to_json()
  with open(f'{path_name_results}model_{n_time_steps}.json', "w") as json_file:
    json_file.write(model_json)

  # serialize weights to HDF5
  model.save_weights(f'{path_name_results}model_{n_time_steps}.h5')
  print("Saved model to disk")


In [11]:
def gera_resultado(y_test, predict,nm_dataset, resultado, n_time_steps, Duracao):
 #Mean Squared Error (Mean Squared Difference Between Estimated Values and Actual Values) - MSE
 MSE = mean_squared_error(y_test, predict)    
 #Square Root of Mean Error - RMSE
 RMSE = np.sqrt(mean_squared_error(y_test, predict))    
 #Mean Absolute Distance or Mean Absolute Error - MAE
 MAE= median_absolute_error(y_pred=predict, y_true = y_test) 
  
 #Calculate the MAPE (Mean Absolute Percentage Error)
 MAPE = ((np.mean(np.abs(y_test -predict) / (y_test)))) * 100   
  
 sMAPE = round(
 	np.mean(
 		np.abs(predict - y_test) /
 		((np.abs(predict) + np.abs(y_test)))
 	)*100, 2
 ) 
 salvar_resultado(nm_dataset, resultado, n_time_steps, MSE, RMSE, MAE, MAPE, sMAPE, Duracao)

In [12]:
def previsao_LSTM(nm_dataset, dataset, n_time_steps, l1, l2, l3, num_epochs, batch_size): 
    
    # Convert dataset to numpy array
    if isinstance(dataset, pd.DataFrame):
        dataset = dataset['num_observations'].values.astype('float32')
    else:
        dataset = np.array(dataset, dtype='float32')
    
    # Handle n_time_steps = 0
    if n_time_steps == 0:
        # For window zero, use only current value as feature
        nlinhas = int(len(dataset) * 0.80)
        test = dataset[nlinhas:len(dataset)]  
        train = dataset[0:nlinhas]
        
        # Check if we have enough data for the dummy features
        if len(train) <= 1 or len(test) <= 1:
            print(f"Skipping n_time_steps={n_time_steps}: train size={len(train)}, test size={len(test)}")
            return
        
        # Create dummy features
        X_train = np.ones((len(train) - 1, 1))
        X_test = np.ones((len(test) - 1, 1))
        
        Y_train = train[1:].reshape(-1, 1)
        Y_test = test[1:].reshape(-1, 1)
        
        # Reshape for LSTM [samples, time steps, features]
        X_train = X_train.reshape(X_train.shape[0], X_train.shape[1], 1)
        X_test = X_test.reshape(X_test.shape[0], X_test.shape[1], 1)
        
    else:
        # Original code for n_time_steps > 0
        nlinhas = int(len(dataset) * 0.80)
        test = dataset[nlinhas:len(dataset)]  
        train = dataset[0:nlinhas] 
        
        # Check if we have enough data for the dummy features
        if len(train) <= 1 or len(test) <= 1:
            print(f"Skipping n_time_steps={n_time_steps}: train size={len(train)}, test size={len(test)}")
            return
        
        X_train, Y_train = create_matrix_dataset(train, n_time_steps)
        X_test, Y_test = create_matrix_dataset(test, n_time_steps) 
        
        # Reshape for LSTM [samples, time steps, features]
        X_train = X_train.reshape(X_train.shape[0], X_train.shape[1], 1)
        X_test = X_test.reshape(X_test.shape[0], X_test.shape[1], 1)
    
    # Apply MinMaxScaler
    scaler_X = MinMaxScaler(feature_range=(0, 1))
    scaler_y = MinMaxScaler(feature_range=(0, 1))
    
    # Reshape for scaling
    original_shape_X = X_train.shape
    X_train_flat = X_train.reshape(-1, 1)
    X_train_scaled = scaler_X.fit_transform(X_train_flat).reshape(original_shape_X)
    X_test_scaled = scaler_X.transform(X_test.reshape(-1, 1)).reshape(X_test.shape)
    
    Y_train_scaled = scaler_y.fit_transform(Y_train.reshape(-1, 1)).ravel()
    Y_test_original = Y_test.copy()
      
    
    # Build LSTM model - CORRECTED ARCHITECTURE
    model = Sequential()
    
    if n_time_steps > 0:
        # Architecture for n_time_steps > 0
        model.add(LSTM(l1, input_shape=(n_time_steps, 1), return_sequences=True))
        model.add(LSTM(l2, return_sequences=True))
        model.add(LSTM(l3))
    else:
        # Architecture for n_time_steps == 0
        # Input shape is (1, 1) because we have 1 time step with 1 feature
        model.add(LSTM(l1, input_shape=(1, 1), return_sequences=True))
        model.add(LSTM(l2, return_sequences=False))  # return_sequences=False for last LSTM layer
        # No third LSTM layer for n_time_steps == 0 to maintain compatibility
    
    model.add(Dense(1))
    model.compile(loss='mean_squared_error', optimizer='adam')
    

    # Stops training when loss stops improving
    early_stop = EarlyStopping(
        monitor='loss',           # monitors training loss
        patience=20,              # waits 20 epochs before stopping
        restore_best_weights=True, # reverts to the best model found
        verbose=0,                # prints message when stopping
        min_delta=0.0001          # minimum change to qualify as improvement
    )
    

    Hora_Inicio = time.time()
    
    resultado = f"LSTM ({l1},{l2},{l3}) n_time_steps={n_time_steps} epochs={num_epochs}"
    print(resultado)

    # Train model with early_stop
    model.fit(
        X_train_scaled, Y_train_scaled,
        epochs=num_epochs,        
        batch_size=batch_size,
        verbose=0,                
        shuffle=False,            
        callbacks=[early_stop]
    )
    
    # Predict
    predict_scaled = model.predict(X_test_scaled, batch_size=batch_size, verbose=0)
    
    # Inverse transform predictions
    predict = scaler_y.inverse_transform(predict_scaled)
    
    Hora_Fim = time.time()
    Duracao = Hora_Fim - Hora_Inicio
    
    # Calculate metrics with original scale values
    gera_resultado(Y_test_original.reshape(-1, 1), predict, 
                   nm_dataset, resultado, n_time_steps, Duracao)

In [13]:
        
#create file to results
criar_arquivo_resultado()

print('forecast for IBM Stock prices')
num_epochs = 200 # number of epochs for train
batch_size = 32

#l1, l2, l3 = 8, 18, 8    # 90.6444863489101, 86.26, 72.7875263690948

#for n_time_steps in range(1,25): #predict with 1 to 24 past values of medition 
#    previsao_LSTM('IBM', dataset, n_time_steps, l1, l2, l3,num_epochs, batch_size)


def random_model():
  for n_time_steps in [0, 3, 6, 9, 12, 15, 18, 21, 24]: #range(0,25): #predict with 0 to 24 past values of medition    
    for l1 in [20, 60, 100]: # chose layer 1 nodes - min 8 and max 100
        for l2 in [20, 60, 100]: # chose layer 2 nodes - min 8 and max 100
            for l3 in [20, 60, 100]: # chose layer 3 nodes - min 8 and max 100
                previsao_LSTM('IBM', dataset, n_time_steps, l1, l2, l3,num_epochs, batch_size)            
                
random_model()                

forecast for IBM Stock prices


LSTM (20,20,20) n_time_steps=0 epochs=200


['Dataset', 'Best Params', 'n_time_steps', 'MSE', 'RMSE', 'MAE', 'MAPE', 'sMAPE', 'Duration']
['IBM', 'LSTM (20,20,20) n_time_steps=0 epochs=200', 0, 2906.4995, 53.91196, 15.08239, 17.011652886867523, 9.56, 19.20990753173828]


LSTM (20,20,60) n_time_steps=0 epochs=200


['Dataset', 'Best Params', 'n_time_steps', 'MSE', 'RMSE', 'MAE', 'MAPE', 'sMAPE', 'Duration']
['IBM', 'LSTM (20,20,60) n_time_steps=0 epochs=200', 0, 2896.348, 53.81773, 15.368904, 17.076541483402252, 9.58, 20.141799211502075]


...

['Dataset', 'Best Params', 'n_time_steps', 'MSE', 'RMSE', 'MAE', 'MAPE', 'sMAPE', 'Duration']
['IBM', 'LSTM (100,100,20) n_time_steps=24 epochs=200', 24, 368.43613, 19.19469, 5.3949203, 5.7768456637859344, 2.97, 166.97490000724792]


LSTM (100,100,60) n_time_steps=24 epochs=200


['Dataset', 'Best Params', 'n_time_steps', 'MSE', 'RMSE', 'MAE', 'MAPE', 'sMAPE', 'Duration']
['IBM', 'LSTM (100,100,60) n_time_steps=24 epochs=200', 24, 258.6393, 16.082268, 4.6473923, 4.8754531890153885, 2.5, 312.0476038455963]


LSTM (100,100,100) n_time_steps=24 epochs=200


['Dataset', 'Best Params', 'n_time_steps', 'MSE', 'RMSE', 'MAE', 'MAPE', 'sMAPE', 'Duration']
['IBM', 'LSTM (100,100,100) n_time_steps=24 epochs=200', 24, 348.1873, 18.659777, 5.2729797, 5.610325187444687, 2.88, 190.68054246902466]
